In [0]:

# =============================================================================
# Notebook: 03_bronze_to_silver_aqe
# Purpose: Reads incremental Bronze data, leverages default Serverless AQE,
#          enriches data via joins, and performs SIT Count Checks.
# =============================================================================

from pyspark.sql import functions as F

storage_account = "stsupplychaindata"
bronze_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/inventory_shipments/"
silver_path = f"abfss://silver@{storage_account}.dfs.core.windows.net/enriched_shipments/"

# ---- Step 1: AQE (Adaptive Query Execution) Note ----
# Note: On Databricks Serverless, AQE is enabled by default and optimized by 
# the underlying engine. Manual overrides are restricted by the platform.
print("Leveraging Databricks Serverless built-in AQE for runtime optimization.")

# ---- Step 2: Read Bronze Data ----
df_bronze = spark.read.format("delta").load(bronze_path)

# ---- Step 3: Create Mock Enrichment Data ----
# Simulating a dimension table to join with our shipments
warehouses = [("WH-15", "New York", "Active"), ("WH-26", "London", "Active"), 
              ("WH-41", "Tokyo", "Active"), ("WH-45", "Sydney", "Active"),
              ("WH-44", "Berlin", "Maintenance")]
df_wh_dim = spark.createDataFrame(warehouses, ["warehouse_id", "location", "status"])

# ---- Step 4: Transform & Enrich (Triggering AQE during the Join) ----
print("\n--- Applying Transformation & Enrichment Logic ---")
df_enriched = df_bronze.join(df_wh_dim, on="warehouse_id", how="left")

# Write to Silver
df_enriched.write.mode("overwrite").format("delta").save(silver_path)
print("Data Enriched and Saved to Silver Layer.")

# ---- Step 5: SIT Testing (Source to Target Count Check) ----
print("\n=======================================================")
print("SIT / ETL TESTING: COUNT CHECK (BRONZE vs SILVER)")
print("=======================================================")

bronze_count = df_bronze.count()
silver_count = spark.read.format("delta").load(silver_path).count()

print(f"Total Records in Bronze (Source) : {bronze_count}")
print(f"Total Records in Silver (Target) : {silver_count}")

if bronze_count == silver_count:
    print("COUNT CHECK PASSED: No data loss during transformation.")
else:
    print(f"COUNT CHECK FAILED: Mismatch detected!")

Leveraging Databricks Serverless built-in AQE for runtime optimization.

--- Applying Transformation & Enrichment Logic ---
Data Enriched and Saved to Silver Layer.

SIT / ETL TESTING: COUNT CHECK (BRONZE vs SILVER)
Total Records in Bronze (Source) : 1000
Total Records in Silver (Target) : 1000
COUNT CHECK PASSED: No data loss during transformation.
